# Лабораторная работа №3
## Тема: Создание и обучение сети на базе LSTM для генерации текста



## 1. Предобработка данных и токенизация
Переводим текст в нижний регистр, извлекаем уникальные символы, строим карты индексов и нарезаем текст на скользящие последовательности. Используем целочисленное кодирование для экономии памяти.

In [7]:
import tensorflow as tf
import numpy as np
import random

with open('lab_3_input_text.txt', 'r', encoding='utf-8') as f:
    text = f.read().lower()

chars = sorted(list(set(text)))
char_to_index = {c: i for i, c in enumerate(chars)}
index_to_char = {i: c for i, c in enumerate(chars)}
print(f"Количество уникальных символов: {len(chars)}")

maxlen = 40
step = 3
sentences = []
next_chars = []

for i in range(0, len(text) - maxlen, step):
    sentences.append(text[i : i + maxlen])
    next_chars.append(text[i + maxlen])
print(f"Создано обучающих последовательностей: {len(sentences)}")

# Оптимизировано: используем int32 вместо огромных bool матриц
X = np.zeros((len(sentences), maxlen), dtype=np.int32)
y = np.zeros((len(sentences),), dtype=np.int32)

for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        X[i, t] = char_to_index[char]
    y[i] = char_to_index[next_chars[i]]

print(f"Формат тензора X (входные данные): {X.shape}")
print(f"Формат тензора y (целевые ответы): {y.shape}")

Количество уникальных символов: 100
Создано обучающих последовательностей: 394621
Формат тензора X (входные данные): (394621, 40)
Формат тензора y (целевые ответы): (394621,)


---
## 2. Архитектура нейросети LSTM
Спроектируем рекуррентную модель с явным слоем `Input` (для Keras 3) и слоем `Embedding` для эффективной обработки символов.

In [8]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(maxlen,)),
    tf.keras.layers.Embedding(input_dim=len(chars), output_dim=64),
    tf.keras.layers.LSTM(128),
    tf.keras.layers.Dense(len(chars), activation='softmax')
])

# Используем sparse_categorical_crossentropy для целочисленных таргетов
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 40, 64)         │         6,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100)            │        12,900 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 118,116 (461.39 KB)

 Trainable params: 118,116 (461.39 KB)

 Non-trainable params: 0 (0.00 B)

---
## 3. Обучение модели
Запускаем обучение сети. Благодаря оптимизации памяти, процесс пройдет быстро и без предупреждений аллокатора.

In [9]:
print("Запуск обучения модели...")
EPOCHS = 5
history = model.fit(X, y, batch_size=128, epochs=EPOCHS, verbose=1)

print(f"Обучение завершено. Финальный Loss: {history.history['loss'][-1]:.4f}")

Запуск обучения модели...
Epoch 1/5


W0000 00:00:1780330220.800164   67207 cpu_allocator_impl.cc:82] Allocation of 63139360 exceeds 10% of free system memory.


3083/3083 ━━━━━━━━━━━━━━━━━━━━ 115s 37ms/step - loss: 2.4549
Epoch 2/5
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 113s 37ms/step - loss: 2.1083
Epoch 3/5
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 110s 36ms/step - loss: 1.9581
Epoch 4/5
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 108s 35ms/step - loss: 1.8672
Epoch 5/5
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 130s 42ms/step - loss: 1.8063
Обучение завершено. Финальный Loss: 1.8063


---
## 4. Результат генерации текста (Абракадабра)
Генерируем текст на основе случайного или заданного фрагмента исходного текста.

In [11]:
def sample(preds, temperature=1.0):
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds + 1e-7) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

# Автоматический поиск фразы-затравки
search_seed = "топор"
start_index = text.find(search_seed)

if start_index == -1:
    # Если фраза не найдена, берем случайный кусок
    start_index = random.randint(0, len(text) - maxlen - 1)
    
sentence = text[start_index: start_index + maxlen]
generated = sentence

print(f'--- Начальная фраза (Seed): "{sentence}"\n')

for i in range(200):
    x_pred = np.zeros((1, maxlen), dtype=np.int32)
    for t, char in enumerate(sentence):
        if char in char_to_index:
            x_pred[0, t] = char_to_index[char]

    preds = model.predict(x_pred, verbose=0)[0]
    next_index = sample(preds, temperature=0.6)
    next_char = index_to_char[next_index]

    generated += next_char
    sentence = sentence[1:] + next_char

print("--- СГЕНЕРИРОВАННАЯ АБРАКАДАБРА:")
print(generated)

--- Начальная фраза (Seed): "топором ее, чего! покончить с ней разом,"

--- СГЕНЕРИРОВАННАЯ АБРАКАДАБРА:
топором ее, чего! покончить с ней разом, как бы она подимотился и всегда начинав вздригайлов и подиминовным леблю в господин, применя и сказать вчера от всем на раскольников.

— ведь он приперталась сториони романа рассуда бы принцу даже пр
